In [ ]:
# Se cargan las canastas de pólizas para identificar productos que aparecen juntos.

import pandas as pd

purchases = pd.read_csv("../data/policy_baskets.csv")
baskets = purchases.groupby("customer_id")["product"].agg(set)
baskets.head()

In [ ]:
# ¿Qué combinaciones de dos productos tienen soporte suficiente para analizar?

from itertools import combinations

products = sorted(purchases["product"].unique())
total_baskets = len(baskets)
pair_supports = []
for pair in combinations(products, 2):
    count = sum(set(pair).issubset(basket) for basket in baskets)
    pair_supports.append({"items": " | ".join(pair), "basket_count": count, "support": count / total_baskets})
pair_supports = pd.DataFrame(pair_supports)
pair_supports

In [ ]:
# Se calculan confianza y lift para evitar recomendar solo productos frecuentes.

product_support = purchases.groupby("product")["customer_id"].nunique() / total_baskets
rules = []
for _, pair in pair_supports.query("support >= 0.15").iterrows():
    left, right = pair["items"].split(" | ")
    for antecedent, consequent in [(left, right), (right, left)]:
        confidence = pair["support"] / product_support[antecedent]
        lift = confidence / product_support[consequent]
        rules.append({"antecedent": antecedent, "consequent": consequent, "support": pair["support"], "confidence": confidence, "lift": lift, "basket_count": pair["basket_count"]})
rules = pd.DataFrame(rules).query("confidence >= 0.50 and lift >= 1.10").sort_values("lift", ascending=False)
rules

In [ ]:
# Se conservan las reglas filtradas para discutir recomendaciones de cross-sell.

from pathlib import Path

submission_dir = Path("../submission")
rules.to_csv(submission_dir / "cross_sell_rules.csv", index=False)